In [1]:
import numpy as np
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from tensorflow.keras.datasets import mnist          # importing dataset

In [2]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape)
print(x_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
(60000, 28, 28)
(10000, 28, 28)


In [3]:
x_train = x_train.reshape(-1, 28*28).astype('float32') / 255.0            # flattening the dimensions into a 2D vector, changing the data type and scaling the values
x_test = x_test.reshape(-1, 28*28).astype('float32') / 255.0
# transforms the training images from 2D matrices into 1D vectors so they can be passed into a standard dense (fully connected) neural network layer


print(x_train.shape)
print()
x_train

(60000, 784)



array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)

#### Original Shape Before Reshaping
When MNIST data is loaded with `mnist.load_data()`, `x_train` is a 3D NumPy array:
* **Shape:** `(60000, 28, 28)`
* **Meaning:** 60,000 images, where each image is a 2D grid of 28 × 28 pixels.


#### What `reshape(-1, 28*28)` Does
* **`28*28` (= 784):** 
  Flattens each 2D image (28 rows × 28 columns) into a single 1D row vector of 784 continuous pixel values.
* **`-1` (automatic dimension):** 
  In NumPy, passing `-1` instructs it to infer the size of that dimension automatically based on the total number of elements:
  $$\frac{60000 \times 28 \times 28}{784} = 60000$$
* **New Shape:** `(60000, 784)`
  Now it is a 2D matrix where each row represents one full image flattened into 784 features.

#### `.astype('float32')` (Precision Casting)
* **What it does:** Converts the array data type from `uint8` (unsigned 8-bit integer, values 0 to 255) to `float32` (32-bit floating-point).
* **Why it's necessary:**
  * **Avoids Integer Division:** Dividing `uint8` directly could cause truncation or unexpected typing issues depending on the environment.
  * **Framework Standard:** TensorFlow / Keras models calculate weights, biases, and gradients in `float32` by default. Using 32-bit floats provides an ideal balance between precision and computational speed on both CPU and GPU.


#### `/ 255.0` (Feature Normalization / Scaling)
* **What it does:** Divides every pixel value by 255.0, scaling pixel intensities from **`[0, 255]`** down to **`[0.0, 1.0]`**.
* **Why it's critical for Neural Networks:**
  * **Faster Convergence:** Gradient-based optimizers (like SGD, Adam) converge significantly faster and more reliably when input features reside on a small, uniform scale.
  * **Prevents Vanishing / Exploding Gradients:** Large raw input values ($0$ to $255$) can produce massive activations, saturating activation functions (like Sigmoid or ReLU) and leading to unstable weight updates.


#### Why is This Done?
* Standard **Dense / Fully Connected layers** (e.g., `layers.Dense(...)`) expect a 1D vector of features for each sample (shape: `(batch_size, num_features)`), rather than a 2D grid.
* Flattening turns each 2D image into 784 input nodes for the neural network.


#### The Second Line (`x_train`)
In Jupyter Notebooks, writing a variable name by itself on the last line of a cell instructs IPython to display its representation and metadata (outputting `shape=(60000, 784)`).

### Sequential API

In [19]:
# Sequential API (used for one input to one ouput)

# STEP 1
model = keras.Sequential([
    keras.Input(shape=(784,)),                            # This step is for only if the data is already flattened
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10)    
])

# STEP 2
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(0.001),
    metrics=['accuracy']
)

# STEP 3
model.fit(x_train, y_train,
          batch_size=64,
          epochs=5,
          verbose=2
         )

# STEP 4
model.evaluate(x_test, y_test,
              batch_size=64,
              verbose=2
              )



Epoch 1/5
938/938 - 4s - 4ms/step - accuracy: 0.9406 - loss: 0.1986
Epoch 2/5
938/938 - 2s - 2ms/step - accuracy: 0.9751 - loss: 0.0784
Epoch 3/5
938/938 - 2s - 2ms/step - accuracy: 0.9837 - loss: 0.0506
Epoch 4/5
938/938 - 2s - 2ms/step - accuracy: 0.9876 - loss: 0.0383
Epoch 5/5
938/938 - 2s - 2ms/step - accuracy: 0.9905 - loss: 0.0291
157/157 - 1s - 6ms/step - accuracy: 0.9803 - loss: 0.0707


[0.07073904573917389, 0.9803000092506409]

In [20]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Sequential API (used for one input to one ouput)

# STEP 1
model = keras.Sequential([
    layers.InputLayer(shape=(28,28)),                   # since the shape of x_train is in (60000, 28, 28)
    layers.Flatten(),                                   # follow this code and above code if the data is not flattened, but the data has to be normalized / scaled [but not reshaped or flattened]
    layers.Dense(512, activation='relu'),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activatin='softmax')    
])

# STEP 2
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(0.001),
    metrics=['accuracy']
)

# STEP 3
model.fit(x_train, y_train,
          batch_size=64,
          epochs=5,
          verbose=2
         )

# STEP 4
model.evaluate(x_test, y_test,
              batch_size=64,
              verbose=2
              )

Epoch 1/5
938/938 - 4s - 4ms/step - accuracy: 0.9408 - loss: 0.1974
Epoch 2/5
938/938 - 2s - 2ms/step - accuracy: 0.9761 - loss: 0.0785
Epoch 3/5
938/938 - 2s - 2ms/step - accuracy: 0.9834 - loss: 0.0512
Epoch 4/5
938/938 - 2s - 2ms/step - accuracy: 0.9870 - loss: 0.0400
Epoch 5/5
938/938 - 2s - 2ms/step - accuracy: 0.9911 - loss: 0.0284
157/157 - 1s - 6ms/step - accuracy: 0.9750 - loss: 0.0883


[0.08834410458803177, 0.9750000238418579]

### Functional API

In [24]:
# using Functional API
# Functional API can handle --
    # One Input -  One Output (with skip connections/branching like ResNet)
    # Multiple Inputs - One Output
    # One Input - Multiple Outputs
    # Multiple Inputs - Multiple Outputs

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# STEP 1
inputs = keras.Input(shape=(28,28))

x = layers.Flatten()(inputs)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dense(256, activation='relu')(x)

outputs = layers.Dense(10, activation='softmax')(x)

model = keras.Model(inputs=inputs, outputs=outputs)


# STEP 2
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(0.001),
    metrics=['accuracy']
)


# STEP 3
model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=5,
    verbose=2
)


# STEP 4
test_loss, test_acc = model.evaluate(
    x_test, y_test,
    batch_size=64,
    verbose=2
)


Epoch 1/5
938/938 - 4s - 5ms/step - accuracy: 0.9402 - loss: 0.1982
Epoch 2/5
938/938 - 2s - 2ms/step - accuracy: 0.9759 - loss: 0.0782
Epoch 3/5
938/938 - 2s - 2ms/step - accuracy: 0.9834 - loss: 0.0513
Epoch 4/5
938/938 - 2s - 2ms/step - accuracy: 0.9887 - loss: 0.0362
Epoch 5/5
938/938 - 2s - 2ms/step - accuracy: 0.9898 - loss: 0.0304
157/157 - 1s - 6ms/step - accuracy: 0.9760 - loss: 0.0851


In [25]:
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Test Accuracy: 97.60%


In [26]:
model.predict(x_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


array([[ -5.226107  ,  -0.8547913 ,  -2.7302976 , ...,  16.714575  ,
         -7.289047  ,  -0.33035907],
       [ -2.7054403 ,   3.6082044 ,  22.782427  , ...,   0.36950213,
         -0.5879135 , -17.236942  ],
       [ -8.256545  ,  12.920335  ,  -5.8930116 , ...,   1.8211432 ,
         -1.5518804 ,  -6.9369507 ],
       ...,
       [-18.199389  ,  -1.1909074 , -11.169789  , ...,   3.7463741 ,
         -5.1737933 ,   8.06912   ],
       [ -5.6791525 ,  -8.050046  ,  -7.7464876 , ...,  -1.985421  ,
          5.3431306 ,  -6.72873   ],
       [ -3.0951521 ,  -9.559446  , -10.392708  , ..., -15.014616  ,
         -4.0810814 ,  -4.5265784 ]], dtype=float32)